# Temporal Patterns — The Arbitrage Cycle

Pumped storage operators exploit the intraday price spread. This notebook dissects the temporal structure of the target to understand:

1. **Intraday cycle** — when does the system pump vs generate?
2. **Weekday vs weekend** — does reduced industrial demand change the pattern?
3. **Seasonal variation** — how does the cycle shift between summer (high solar) and winter (high demand)?
4. **Regime switching** — what fraction of time is spent pumping vs generating at each hour?

Back to [main EDA](eda.ipynb)

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd

from src.data import load_train
from src.features import add_time_features
from src.plotting import *

train = load_train()
train_t = add_time_features(train)

## 1. Average Hourly Profile

The mean hourly profile reveals the core dispatch pattern. The fill area shows where the system is a net generator vs net consumer on average.

In [2]:
plot_hourly_profile(train).show()

## 2. Hourly Profile by Weekday

Weekdays have sharper swings (industrial load). Weekends are flatter — less demand, less need for peak generation. Saturday and Sunday show noticeably less pumping overnight, suggesting operators reduce activity when weekend price spreads are narrower.

In [3]:
plot_hourly_profile(train, groupby='weekday').show()

In [4]:
plot_weekly_profile(train).show()

## 3. Hourly Profile by Month

Seasonal effects are visible:
- **Summer** (Jun–Aug): Strong midday solar pushes the system toward pumping around 12:00–15:00 — the "solar duck curve" effect.
- **Winter** (Dec–Feb): Higher demand and less solar sharpen the evening generation peak.
- **Shoulder months** (Mar–May, Sep–Nov): Transitional patterns.

In [5]:
plot_hourly_profile(train, groupby='month').show()

## 4. Heatmaps

Two complementary views:
- **Hour × Weekday** — shows the weekday/weekend modulation of the intraday cycle.
- **Hour × Month** — shows the seasonal modulation. Look for the midday pumping pocket in summer months.

In [6]:
plot_hourly_heatmap(train).show()

In [7]:
plot_monthly_hourly_heatmap(train).show()

## 5. Regime Analysis

Left panel: what fraction of hours at each time of day are spent pumping vs generating.  
Right panel: conditional mean intensity — given that the system *is* generating, how hard is it generating?

Note the 03:00–05:00 window: nearly 80%+ of hours are pumping. Conversely, 19:00–21:00 is ~70%+ generation.

In [8]:
plot_regime_analysis(train).show()

## 6. Distribution Shape by Hour and Month

Violin plots show the full conditional distribution. At 03:00, the distribution is heavily skewed negative (pumping). At 19:00, heavily positive (generation). But there is overlap at every hour — the system can be in either regime at any time.

In [9]:
plot_violin(train_t, 'es_total_ps', groupby_col='hour',
            title='Target Distribution by Hour of Day').show()

In [10]:
plot_monthly_profile(train, show_box=True).show()

## 7. Volatility Profile

Standard deviation by hour — when is the target hardest to predict? Transition hours (07:00–09:00 and 16:00–18:00) have the highest variance, exactly when the system is switching regimes.

In [11]:
plot_hourly_profile(train, agg='std',
                    ).show()

---

## Modelling Implications

1. **Hour of day is the single most important feature.** Any model must capture the intraday cycle. Consider using hour as a categorical variable (one-hot or cyclic encoding) rather than a continuous integer.

2. **Weekday/weekend interaction matters.** A simple `is_weekend` flag may capture most of the effect, but allowing hour × weekday interactions could improve peak-hour predictions.

3. **Seasonal modulation.** Month or a cyclic seasonal feature should interact with hour. The summer midday pumping pocket is a distinct regime not present in winter.

4. **Regime transition hours (07–09, 16–18) are where errors concentrate.** Extra modelling effort here (e.g., separate models for transition hours) could reduce RMSE.